# 15 · Backtest con el reforecast GEFSv12 (2000-2019)

Pregunta: con la **lluvia pronosticada** (no la observada), ¿qué tan bien se anticipa un día con huaico catalogado en Chosica-Chaclacayo-Santa Eulalia-Ricardo Palma, a 24, 48 y 72 horas, a lo largo de 20 temporadas?

> **No es un sistema de alerta ni una predicción para decidir evacuaciones.** Es un banco de pruebas. Ante una emergencia llama al 115 (INDECI).

**Pre-registro** (escrito antes de mirar ningún puntaje):
- **T40** con la lluvia del reforecast reescalada por cuantiles, el modelo entrenado con ERA5 (PP) conserva a 24 h al menos el 70 % de la ganancia de precisión media sobre la climatología que tiene con la lluvia ERA5 («techo»), y esa fracción baja con el plazo.
- **T41** un MOS entrenado dentro de 2000-2019 con estadísticos del ensamble no supera al PP con ERA5 (solo 20 temporadas).
- **T42** la probabilidad de excedencia del ensamble y su dispersión aportan sobre la media.
- **T43** el resultado depende de 2017 (23 de 56 días con huaico).
- **T44** CAPE y agua precipitable del reforecast (solo miembro de control) aportan sobre la lluvia (T29).
Cajas fijas de antemano: `valle` (las 6 celdas de PISCOp), y las de `fuentes.CAJAS_CHIRPS`. Plazo k: corrida de las 00 UTC emitida k-1 días antes (docs/19 §2).

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd

from illapa_research import gefs_backtest as B
from illapa_research import gefs_reforecast as G
from illapa_research import modelo_diario as md

pd.set_option("display.width", 200)
RAIZ = Path("..") if Path("../datos").exists() else Path(".")
N_NULAS = int(os.environ.get("ILLAPA_NULAS", "2000"))
corridas = G.cargar_corridas(RAIZ / "datos" / "gefs_reforecast")
print("corridas:", corridas["run"].nunique(), "| primera", corridas["run"].min().date(), "| última", corridas["run"].max().date())
print("miembros:", sorted(corridas["miembro"].unique()), "| variables:", sorted(corridas["variable"].unique()))

## 1. Datos y alineación

Cada corrida (00 UTC) trae 5 miembros, rejilla de 0,25° y ventanas de 6 horas. La lluvia de un día local de Lima es la suma de 4 ventanas (horas 6-30 de la corrida del mismo día para el plazo 1, 30-54 de la del día anterior para el 2, 54-78 de la de dos días antes para el 3).

In [ ]:
D = md.cargar_datos(RAIZ)
print(D.inventario["puntos_era5"], "|", D.inventario["ultimo_dia"])
R = B.correr(D, corridas, plazos=(1, 2, 3), n_nulas=N_NULAS)
print("días de prueba:", len(R["dias"]), "| con huaico:", int(R["y"].sum()), "| sin 2017:", int(R["y"][R["ts"] != 2017].sum()))
pd.DataFrame({"eventos": pd.Series(R["y"]).groupby(R["ts"]).sum(), "días": pd.Series(R["y"]).groupby(R["ts"]).size()}).T

## 2. Resultados por plazo (todas las temporadas y sin 2017)

AP = precisión media (la base es la tasa de días con huaico); «aciertos 5 %» = fracción de los días con huaico que caen dentro del 5 % de días de mayor puntaje. `techo` = el mismo modelo con lluvia ERA5.

In [ ]:
T = B.tabla_metricas(R)
cols = ["plazo", "subconjunto", "modelo", "eventos", "ap", "ap_base", "auc", "aciertos_5pc", "aciertos_10pc", "brier_skill", "obs_sobre_pred"]
orden = ["climatologia", "techo", "pp_qm", "pp_crudo", "pp_qm_sin2017_ent", "mos_media", "mos_ens", "mos_ant", "mos_ens_conv", "gefs_media", "gefs_max_ens"]
for sub in ("todo", "sin_2017"):
    t = T[T.subconjunto == sub][cols].copy()
    t["modelo"] = pd.Categorical(t["modelo"], orden, ordered=True)
    display(t.sort_values(["plazo", "modelo"]).round(3).reset_index(drop=True))

## 3. Intervalos (remuestreo de temporadas) y diferencias pareadas

In [ ]:
filas = []
for k, mod in R["plazos"].items():
    for sub, fn in (("todo", B.SUBCONJUNTOS["todo"]), ("sin_2017", B.SUBCONJUNTOS["sin_2017"])):
        m = fn(R["ts"])
        cols_p = {n: mod[n][m] for n in ("techo", "pp_qm", "mos_ens", "mos_ant", "gefs_media")}
        cols_p["climatologia"] = R["clim"][m]
        bt = B.bootstrap_temporadas(R["y"][m], R["ts"][m], cols_p, n=1000)
        bt.insert(0, "subconjunto", sub); bt.insert(0, "plazo", k)
        filas.append(bt)
display(pd.concat(filas).round(3).reset_index(drop=True))

In [ ]:
filas = []
for k, mod in R["plazos"].items():
    for sub, fn in (("todo", B.SUBCONJUNTOS["todo"]), ("sin_2017", B.SUBCONJUNTOS["sin_2017"])):
        m = fn(R["ts"]); y = R["y"][m]; ts = R["ts"][m]
        for a, b in (("pp_qm", "techo"), ("pp_qm", "pp_crudo"), ("mos_ens", "mos_media"), ("mos_ens", "pp_qm"), ("mos_ens_conv", "mos_ens"), ("mos_ant", "mos_ens")):
            if a in mod and b in mod:
                filas.append({"plazo": k, "subconjunto": sub, "a": a, "b": b, **B.diferencia_pareada(y, ts, mod[a][m], mod[b][m])})
display(pd.DataFrame(filas).round(3))

## 4. Mundos nulos (etiquetas desplazadas dentro de cada temporada, todo el procedimiento reajustado)

In [ ]:
display(B.tabla_nulas(R).round(4))

## 5. Fiabilidad y valor económico (cost-loss)

Bandas de probabilidad del mejor modelo de cada plazo y cuántos días con huaico cayó en cada una.

In [ ]:
from illapa_research.pronostico_probabilidad import bandas_fiabilidad
for k in (1, 2, 3):
    for nombre in ("pp_qm", "mos_ens"):
        print(f"plazo {k} · {nombre}")
        display(bandas_fiabilidad(R["y"], R["plazos"][k][nombre]).round(3))

In [ ]:
for k in (1, 2, 3):
    print("plazo", k)
    v = B.valor_costo_perdida(R["y"], R["plazos"][k]["mos_ens"]).rename(columns={"valor": "mos_ens"})
    v["pp_qm"] = B.valor_costo_perdida(R["y"], R["plazos"][k]["pp_qm"])["valor"].to_numpy()
    v["techo"] = B.valor_costo_perdida(R["y"], R["plazos"][k]["techo"])["valor"].to_numpy()
    display(v.round(3))

## 6. Umbrales en mm: el sesgo de GEFS frente a PISCOp (SENAMHI)

GEFS (ensamble medio, caja del valle) contra la lluvia observada PISCOp v3.0 en las mismas celdas: sesgo, aciertos y falsas alarmas con umbrales de percentiles de PISCOp, sin corregir y corregido por cuantiles fuera de temporada. Una corrección monótona no cambia el orden de los días: solo la calibración.

In [ ]:
from illapa_research import correccion_sesgo as CS
ruta = RAIZ / "datos" / "pisco_valle_diario.csv"
if ruta.exists():
    pisco = CS.serie_pisco(None, ruta)
    huaicos = [d.date() for d in R["dias"][R["y"] == 1]]
    filas = []
    for k in (1, 2, 3):
        gv = B.serie_gefs(R["ens"], k, "valle", "media")
        c = CS.comparar(gv, pisco, huaicos)
        c.insert(0, "plazo", k)
        filas.append(c[c.correccion.isin(["sin corregir", "cuantiles"])][["plazo", "correccion", "percentil_obs", "umbral_mm", "sesgo", "r", "pod", "far", "csi", "capturados", "huaicos", "tasa_alarma_otros"]])
    display(pd.concat(filas).round(3).reset_index(drop=True))
else:
    print("falta datos/pisco_valle_diario.csv (se genera con correccion_sesgo.serie_pisco)")

## 7. ¿Qué quebrada? P(día) × participación, con y sin 2017

In [ ]:
from illapa_research import filtro, integracion as ig, quebradas as q
from illapa_research.catalogo import CATALOGO, cargar_cuencas, cargar_todos
cuencas = filtro.cuencas_a_tabla(cargar_cuencas())
refs = q.cargar_referencias(CATALOGO / "quebradas_referencias.csv")
hist = filtro.asignar_historial(cargar_todos(solo_dia_exacto=True, sin_baja_confianza=True), refs, cuencas)
ranking = filtro.tabla(cuencas, hist, D.temporadas)
ubicadas = sorted(ranking.loc[ranking.evidencia == "fisica+historial", "slug"])
ts_dia = pd.Series(R["ts"], index=R["dias"])
filas = []
for k in (1, 2, 3):
    p_dia = pd.Series(R["plazos"][k]["mos_ens"], index=R["dias"])
    for sub in ("todo", "sin_2017"):
        m = (R["ts"] != 2017) if sub == "sin_2017" else np.ones(len(p_dia), bool)
        r = ig.validar_pares(p_dia[m], hist.activaciones, ubicadas, ts_dia[m], D.temporadas)
        r.insert(0, "subconjunto", sub); r.insert(0, "plazo", k)
        filas.append(r)
display(pd.concat(filas).round(4))
display(pd.DataFrame([B.cobertura_top_k(hist.activaciones, ubicadas, D.temporadas)]).round(3))

## 8. Potencia: cuántos eventos hay y qué se puede detectar

In [ ]:
pot = pd.DataFrame({"temporadas": pd.Series(R["ts"]).groupby(pd.cut(R["ts"], [1999, 2008, 2013, 2019])).nunique(),
                    "días": pd.Series(R["y"]).groupby(pd.cut(R["ts"], [1999, 2008, 2013, 2019])).size(),
                    "días con huaico": pd.Series(R["y"]).groupby(pd.cut(R["ts"], [1999, 2008, 2013, 2019])).sum()})
display(pot)

## Lectura

Las conclusiones y sus límites (5 miembros, 0,25°, catálogo pobre en 2000-2008, 2017 con el 41 % de los días con huaico) están en `docs/19-backtest-gefs-reforecast.md`. Datos: NOAA GEFSv12 reforecast (AWS Open Data), ERA5, catálogo de huaicos del repositorio; PISCOp v3.0 del SENAMHI (leyenda en el capítulo 11).